# ENSO teleconnection (DJF) — SST & precipitation, IFS-FESOM2-SR 3-member ensemble

Regression of **DJF SST** and **DJF total precipitation** onto the standardized
DJF Niño3.4 index, three ensemble members, common period **1980–2014**.

**Layout:** 3 rows (ens1/ens2/ens3) × 2 columns.
- **Column 1 — Precipitation** (`tprate` → mm/day) filled, with **Z500** regression
  **contours overlaid** (positive solid magenta, negative dashed blue, zero thick black).
- **Column 2 — SST** (`avg_tos`, °C) filled, no contours.

Pacific-centred Robinson projection; significance hatching at the 95% level; one
shared colorbar per column.

**DJF convention** (kept from your working notebooks): shift time +1 month, select
months [1,2,3] → true DJF, drop the incomplete first/last winters, annual-mean.

**Sources**
- SST `avg_tos` (ocean, surface, K→°C): FESOM folders, off-by-one ensemble naming.
- Precip `tprate` & Z500 `mz` (atmosphere): ens1 from intake catalogues
  (`2D_…_atmos_avg` for precip, `3D_…_atmos_avg` for Z500); ens2/ens3 from gribscan
  zarr-JSON refs (`sfc.dir/atm2d_avg.json`, `pl.dir/atm3d_avg.json`).

**Self-contained** — recomputes Z500 here. If the 3D atmosphere json for ens2/ens3
isn't ready, the Z500 contour for those rows is skipped with a printed warning; the
precip-fill and SST columns still render.


In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import intake
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
from scipy.stats import t as student_t

def ifs_to_latlon(ds):
    return ds.rename({'value': 'latlon'}).set_index(latlon=("lat", "lon")).unstack("latlon")

def wgt_areaave(indat, latS, latN, lonW, lonE):
    lat = indat.lat; lon = indat.lon
    if ((lonW < 0) or (lonE < 0)) and (lon.values.min() > -1):
        anm = indat.assign_coords(lon=((lon + 180) % 360 - 180))
    else:
        anm = indat
    iplat = lat.where((lat >= latS) & (lat <= latN), drop=True)
    iplon = lon.where((lon >= lonW) & (lon <= lonE), drop=True)
    wgt = np.cos(np.deg2rad(lat))
    return anm.sel(lat=iplat, lon=iplon).weighted(wgt).mean(("lon", "lat"), skipna=True)

def detrend_dim(da, dim="time", deg=1):
    coeffs = da.polyfit(dim=dim, deg=deg)
    return da - xr.polyval(da[dim], coeffs.polyfit_coefficients)

def _resample_year(da):
    for f in ("YE", "Y", "A"):
        try:
            return da.resample(time=f).mean("time")
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid annual resample frequency")

def djf_annual_mean(da):
    """True DJF mean per winter via +1-month shift then months [1,2,3]."""
    if da.time.size == 0:
        raise ValueError("empty time axis into djf_annual_mean")
    da = da.assign_coords(time=da.time.to_index() + pd.DateOffset(months=1))
    da = da.sel(time=da.time.dt.month.isin([1, 2, 3]))
    fy = da.time.dt.year.min().item() + 1
    da = da.sel(time=da.time.dt.year >= fy)
    ly = da.time.dt.year.max().item()
    da = da.sel(time=~((da.time.dt.year == ly) & (da.time.dt.month == 1)))
    return _resample_year(da)


In [2]:
def xr_regression(x, y, dim="time", alternative="two-sided"):
    x, y = xr.align(x, y)
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim); ymean = y.mean(dim)
    xstd = x.std(dim);   ystd = y.std(dim)
    cov = ((x - xmean) * (y - ymean)).sum(dim) / n
    cor = cov / (xstd * ystd)
    slope = cov / (xstd ** 2)
    intercept = ymean - xmean * slope
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor ** 2)
    stderr = slope / tstats
    pval = student_t.sf(np.abs(tstats), n - 2) * 2
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)
    return xr.merge([
        cov.rename("cov").astype(np.float32), cor.rename("cor").astype(np.float32),
        slope.rename("slope").astype(np.float32),
        intercept.rename("intercept").astype(np.float32),
        pval.rename("pvalue").astype(np.float32),
        stderr.rename("stderr").astype(np.float32), n.rename("n").astype(np.int16),
    ])


In [3]:
# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
COMMON = slice("1980", "2014")
NINO34 = dict(latS=-5, latN=5, lonL=190, lonR=240)

# SST (ocean avg_tos): (hist_dir, ssp_dir)  — ssp unused for 1980-2014 but kept for parity
SST_SRC = {
    "ens1": ("/work/bm1344/a270228/Phase1-HIST-FESOM/monthly/2d/avg_tos", None),
    "ens2": ("/work/bm1344/a270228/Phase2-HIST-ENS1-FESOM/monthly/2d/avg_tos", None),
    "ens3": ("/work/bm1344/a270228/Phase2-HIST-ENS2-FESOM/monthly/2d/avg_tos", None),
}

# Atmosphere catalogues for ens1
CAT_HIST = "/work/bm1344/a270228/tco1279-ng5-eerie-production-hist-years-final.yaml"
ATM2D_KEY = "2D_monthly_0.25deg_atmos_avg"   # precip tprate
ATM3D_KEY = "3D_monthly_0.25deg_atmos_avg"   # geopotential mz

# Gribscan refs for ens2/ens3 (HIST)
ATM2D_REF = {
    "ens2": "reference::/work/bm1344/a270228/phase2_hist_ens1/gribscan_1m_REGULARLL/jsons.1975-2014/sfc.dir/atm2d_avg.json",
    "ens3": "reference::/work/bm1344/a270228/phase2_hist_ens2/gribscan_1m_REGULARLL/jsons.1975-2014/sfc.dir/atm2d_avg.json",
}
ATM3D_REF = {
    "ens2": "reference::/work/bm1344/a270228/phase2_hist_ens1/gribscan_1m_REGULARLL/jsons.1975-2014/pl.dir/atm3d_avg.json",
    "ens3": "reference::/work/bm1344/a270228/phase2_hist_ens2/gribscan_1m_REGULARLL/jsons.1975-2014/pl.dir/atm3d_avg.json",
}

G = 9.8
PRECIP_FACTOR = 86400000.0   # kg m-2 s-1 -> mm/day  (matches your notebook's *86400000)


In [4]:
# ----------------------------------------------------------------------
# Loaders / index
# ----------------------------------------------------------------------
def load_sst_field(folder, time_slice):
    ds = xr.open_mfdataset(f"{folder}/avg_tos_*.nc")
    da = ds["avg_tos"]
    if da.ndim == 4:
        da = da[:, 0, :, :]
    return (da - 273.15).sel(time=time_slice)

def nino34_std(folder, time_slice, box=NINO34):
    sst = load_sst_field(folder, time_slice)
    nino = wgt_areaave(sst, box["latS"], box["latN"], box["lonL"], box["lonR"])
    ndt = detrend_dim(djf_annual_mean(nino), dim="time")
    return (ndt - ndt.mean()) / ndt.std()

def open_atmos(member, kind):
    """kind in {'2d','3d'}; returns a lat/lon dataset for the member."""
    if member == "ens1":
        cat = intake.open_catalog(CAT_HIST)
        key = ATM2D_KEY if kind == "2d" else ATM3D_KEY
        return ifs_to_latlon(cat[key].to_dask())
    ref = (ATM2D_REF if kind == "2d" else ATM3D_REF)[member]
    return ifs_to_latlon(xr.open_zarr(ref, consolidated=False))

def precip_djf_detrended(member, time_slice):
    ds = open_atmos(member, "2d")
    tp = ds["tprate"].sel(time=time_slice) * PRECIP_FACTOR   # mm/day
    return detrend_dim(djf_annual_mean(tp), dim="time")

def z500_djf_detrended(member, time_slice):
    ds = open_atmos(member, "3d")
    z = ds["mz"]
    z = z.sel(time=time_slice)
    if "level" in z.coords:
        z = z.sel(level=500)
    elif "plev" in z.coords:
        z = z.sel(plev=50000)
    else:
        raise KeyError(f"no level/plev coord; {list(z.coords)}")
    return detrend_dim(djf_annual_mean(z / G), dim="time")

def fix_lon(reg):
    """Wrap lon to 0-360 and sort, so Pacific-centred maps don't tear."""
    reg = reg.assign_coords(lon=reg.lon % 360).sortby("lon")
    return reg


## Compute regressions for all three members (1980–2014)

SST and precip always computed. Z500 is attempted; if the 3D json for ens2/ens3
isn't ready it's skipped (warning printed) and that row's precip panel simply has
no contour overlay.

In [5]:
members = ["ens1", "ens2", "ens3"]
reg_sst, reg_pr, reg_z = {}, {}, {}

for m in members:
    print(f"--- {m} ---")
    nino_std = nino34_std(SST_SRC[m][0], COMMON)

    # SST regression (align nino time to the field's DJF time via positional reassign)
    sst_field = load_sst_field(SST_SRC[m][0], COMMON)
    sst_djf = detrend_dim(djf_annual_mean(sst_field), dim="time")
    # ensure the index shares the SST DJF time axis
    nino_for_sst = nino_std.assign_coords(time=sst_djf.time)
    reg_sst[m] = fix_lon(xr_regression(nino_for_sst, sst_djf, dim="time"))
    print(f"  SST ok (n={int(reg_sst[m].n.max())})")

    # Precip regression
    pr_djf = precip_djf_detrended(m, COMMON)
    nino_for_pr = nino_std.assign_coords(time=pr_djf.time)
    reg_pr[m] = fix_lon(xr_regression(nino_for_pr, pr_djf, dim="time"))
    print(f"  precip ok (n={int(reg_pr[m].n.max())})")

    # Z500 regression (optional)
    try:
        z_djf = z500_djf_detrended(m, COMMON)
        nino_for_z = nino_std.assign_coords(time=z_djf.time)
        reg_z[m] = fix_lon(xr_regression(nino_for_z, z_djf, dim="time"))
        print(f"  Z500 ok (n={int(reg_z[m].n.max())})")
    except Exception as e:
        reg_z[m] = None
        print(f"  Z500 SKIPPED for {m}: {type(e).__name__}: {str(e)[:80]}")


--- ens1 ---


/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/array/numpy_compat.py:40: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in sqrt
  return func(*(_execute_task(a, cache) for a in args))


  SST ok (n=34)


/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


  precip ok (n=34)
  Z500 ok (n=34)
--- ens2 ---


...

  precip ok (n=34)
  Z500 ok (n=34)
--- ens3 ---


/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/array/numpy_compat.py:40: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in sqrt
  return func(*(_execute_task(a, cache) for a in args))
/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


  SST ok (n=34)


/sw/spack-levante/mambaforge-22.9.0-2-Linux-x86_64-kptncg/lib/python3.10/site-packages/dask/core.py:119: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


  precip ok (n=34)
  Z500 ok (n=34)


## Figure — 3 members × [precip + Z500 contours | SST]

In [ ]:
def add_features(ax):
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="k", linewidth=0.3)
    ax.set_global()

def hatch_sig(ax, reg):
    (1 - reg.pvalue).plot.contourf(
        ax=ax, levels=[0, 0.95, 1], hatches=["", "."], alpha=0,
        transform=ccrs.PlateCarree(), add_colorbar=False)

# Z500 contour levels (m per index std)
Z_LEVELS = np.arange(-40, 41, 8)

fig = plt.figure(figsize=(20, 18))
proj = ccrs.Robinson(central_longitude=180)
nrows = len(members)

# axis grid: 2 columns
left_col, right_col = 0.04, 0.52
width, height = 0.42, 0.26
tops = [0.70, 0.40, 0.10]   # row y-positions

precip_cf = sst_cf = None
for r, m in enumerate(members):
    # ---- column 1: precip filled + Z500 contours ----
    axp = fig.add_axes([left_col, tops[r], width, height], projection=proj)
    precip_cf = reg_pr[m].slope.plot.contourf(
        ax=axp, vmin=-1, vmax=1, levels=21, extend="both",
        cmap=cmo.tarn, add_colorbar=False, transform=ccrs.PlateCarree(), robust=True)
    hatch_sig(axp, reg_pr[m])
    if reg_z[m] is not None:
        z = reg_z[m].slope
        neg = [l for l in Z_LEVELS if l < 0]; pos = [l for l in Z_LEVELS if l > 0]
        if neg:
            cn = z.plot.contour(ax=axp, levels=neg, colors="darkblue",
                                linestyles="--", linewidths=1,
                                transform=ccrs.PlateCarree(), add_colorbar=False)
            axp.clabel(cn, inline=True, fontsize=7, fmt="%.0f")
        if pos:
            cp = z.plot.contour(ax=axp, levels=pos, colors="magenta",
                                linestyles="-", linewidths=1,
                                transform=ccrs.PlateCarree(), add_colorbar=False)
            axp.clabel(cp, inline=True, fontsize=7, fmt="%.0f")
        cz = z.plot.contour(ax=axp, levels=[0], colors="black", linewidths=1.6,
                            transform=ccrs.PlateCarree(), add_colorbar=False)
    add_features(axp)
    axp.set_title(f"{m.upper()}  precip + Z500", fontsize=15)

    # ---- column 2: SST filled ----
    axs = fig.add_axes([right_col, tops[r], width, height], projection=proj)
    sst_cf = reg_sst[m].slope.plot.contourf(
        ax=axs, vmin=-1.5, vmax=1.5, levels=31, extend="both",
        cmap="RdBu_r", add_colorbar=False, transform=ccrs.PlateCarree(), robust=True)
    hatch_sig(axs, reg_sst[m])
    add_features(axs)
    axs.set_title(f"{m.upper()}  SST", fontsize=15)

# shared colorbars (one per column)
cax_p = fig.add_axes([left_col, 0.055, width, 0.015])
cb_p = fig.colorbar(precip_cf, cax=cax_p, orientation="horizontal")
cb_p.set_label("DJF precip regression on Niño3.4 (mm/day per std)", fontsize=13)

cax_s = fig.add_axes([right_col, 0.055, width, 0.015])
cb_s = fig.colorbar(sst_cf, cax=cax_s, orientation="horizontal")
cb_s.set_label("DJF SST regression on Niño3.4 (°C per std)", fontsize=13)

#fig.suptitle("ENSO–SST & precipitation DJF teleconnection (1980–2014), 3 members\n"
             #"precip column overlaid with Z500 regression contours (m per std)",
             #fontsize=18, y=0.95)
fig.savefig("ENSO_SST_precip_Z500_teleconnection_3members_1980-2014.png",
            bbox_inches="tight", dpi=300)
plt.show()


### Tweak points
- **Precip colour range** `vmin/vmax=±1` mm/day per std (matches your notebook). Raise if it saturates.
- **SST range** `±1.5 °C` per std — typical ENSO amplitude; adjust as needed.
- **Z500 contour levels** `np.arange(-40,41,8)`; tighten the step for more contours.
- **ens2/ens3 level coord**: handled automatically (`level`=500 hPa or `plev`=50000 Pa).
- If the SST DJF grid and the atmosphere DJF grid differ in resolution, that's fine —
  each regression is computed on its own native grid; only the Niño index time axis is shared.